# AeroDelay AI - retrained at scale (BTS on-time data + Open-Meteo weather)

**Run all cells top to bottom.** No API keys are needed. Default config downloads 10 months of BTS flights (~300 MB) and hourly weather for ~350 US airports; it takes roughly 25-40 minutes (most of it downloads, weather calls and SHAP). Set `QUICK = True` in the config cell for a ~5 minute smoke test (4 months, bigger airports only, 150k rows).

Honest scope: training covers **US domestic flights only** (BTS reporting carriers, ~350 airports). The app can look up any airport in `airport_master` (OurAirports), but predictions outside the training coverage are flagged as low confidence.

What this notebook does: download -> assemble -> **clean (the cell that was missing from the old notebook; it creates `flights_clean`)** -> features -> time-based split -> baselines and models -> final fit -> SHAP -> save artifacts -> demo predictions with forecast weather.

In [ ]:
%pip -q install timezonefinder pyarrow shap requests gradio

In [ ]:
# ---- Config ----
import os, glob, json, time, zipfile, sqlite3, warnings, joblib
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
QUICK = False                 # True = fast smoke test
WORK = "aerodelay_retrain"; os.makedirs(WORK + "/data/wx", exist_ok=True); os.makedirs(WORK + "/out", exist_ok=True); os.chdir(WORK)
MONTHS = ["2025_10","2025_11","2025_12","2026_1","2026_2","2026_3","2026_4","2026_5","2026_6","2026_7"]
MIN_FLIGHTS = 300             # airports with at least this many flights (origin+destination) get weather and are in the model
SAMPLE = 1_000_000            # rows sampled for training
if QUICK: MONTHS, MIN_FLIGHTS, SAMPLE = MONTHS[-4:], 3000, 150_000
# time-based split: train < 2026-05-01 <= validation < 2026-06-01 <= test (the last two months are never used for fitting/selection)
print(MONTHS, MIN_FLIGHTS, SAMPLE)

In [ ]:
import pandas as pd,numpy as np
BASE33=['departure_hour','departure_minute','departure_hour_float','day_of_week','month','day_of_month','is_weekend','encoded_origin','encoded_destination','encoded_route','encoded_carrier','encoded_part_of_day','estimated_path_length_km','origin_avg_delay_safe','destination_avg_delay_safe','route_avg_delay_safe','carrier_avg_delay_safe','origin_flight_count_safe','destination_flight_count_safe','route_flight_count_safe','carrier_flight_count_safe','temperature_c','feels_like_c','pressure_hpa','humidity_pct','wind_strength','wind_deg','visibility','rain_occurrence','high_wind_flag','rain_flag','low_visibility_flag','weather_severity_score']
EXTRA=['local_departure_hour_float','origin_hour_flights','dest_hour_arrivals','cloud_cover_pct','wind_gust_ms','dest_temperature_c','dest_wind_strength','dest_visibility','dest_rain_occurrence','dest_cloud_cover_pct','dest_weather_severity_score']
EXT=BASE33+EXTRA
assert len(BASE33)==33
TRAIN_END=pd.Timestamp('2026-05-01',tz='UTC'); VAL_END=pd.Timestamp('2026-06-01',tz='UTC')
SMOOTH=20
def split(d):
    t=d.scheduled_departure_utc
    return d[t<TRAIN_END].copy(), d[(t>=TRAIN_END)&(t<VAL_END)].copy(), d[t>=VAL_END].copy()
def build_maps(tr,y='target_delay_minutes'):
    g=tr[y].mean(); M={}
    for name,col in [('origin','origin'),('destination','destination'),('route','route'),('carrier','airline_id')]:
        s=tr.groupby(col)[y].agg(['sum','count'])
        M[name+'_mean_map']=((s['sum']+SMOOTH*g)/(s['count']+SMOOTH)).to_dict()
        M[name+'_count_map']=s['count'].to_dict()
    M['global_train_delay_mean']=float(g); return M
def apply_safe(df,M,oof=None):
    df=df.copy(); g=M['global_train_delay_mean']
    for name,col in [('origin','origin'),('destination','destination'),('route','route'),('carrier','airline_id')]:
        df[name+'_avg_delay_safe']=df[col].map(M[name+'_mean_map']).fillna(g)
        df[name+'_flight_count_safe']=df[col].map(M[name+'_count_map']).fillna(0)
    return df
def oof_safe(tr,k=5,seed=0):
    """out-of-fold target encodings for training rows (a row never sees its own delay)"""
    tr=tr.copy(); rng=np.random.RandomState(seed); fold=rng.randint(0,k,len(tr)); out=[]
    full=build_maps(tr)
    for f in range(k):
        m=fold==f; Mf=build_maps(tr[~m]); part=apply_safe(tr[m],Mf)
        for name in ['origin','destination','route','carrier']:  # counts: use full-train counts (no target info)
            col={'carrier':'airline_id'}.get(name,name); part[name+'_flight_count_safe']=part[col].map(full[name+'_count_map']).fillna(0)
        out.append(part)
    return pd.concat(out).loc[tr.index],full
def encodings(tr):
    return {'origin_map':{k:i for i,k in enumerate(sorted(tr.origin.unique()))},'destination_map':{k:i for i,k in enumerate(sorted(tr.destination.unique()))},
            'route_map':{k:i for i,k in enumerate(sorted(tr.route.unique()))},'carrier_map':{k:i for i,k in enumerate(sorted(tr.airline_id.unique()))},
            'part_of_day_map':{k:i for i,k in enumerate(sorted(tr.part_of_day.unique()))}}
def apply_enc(df,E):
    df=df.copy()
    for c,m,k in [('encoded_origin','origin_map','origin'),('encoded_destination','destination_map','destination'),('encoded_route','route_map','route'),('encoded_carrier','carrier_map','airline_id'),('encoded_part_of_day','part_of_day_map','part_of_day')]:
        df[c]=df[k].map(E[m]).fillna(-1).astype(int)
    return df

## 1. Download BTS On-Time Performance (public, no key)

In [ ]:
BTS = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{}.zip"
for ym in MONTHS:
    f = f"data/bts_{ym}.zip"
    if os.path.exists(f) and os.path.getsize(f) > 5_000_000: continue
    r = requests.get(BTS.format(ym), stream=True, timeout=600); r.raise_for_status()
    with open(f, "wb") as fh:
        for chunk in r.iter_content(1 << 20): fh.write(chunk)
    print("downloaded", ym, round(os.path.getsize(f) / 1e6), "MB")

In [ ]:
cols = ["FlightDate","Reporting_Airline","Flight_Number_Reporting_Airline","Origin","Dest","CRSDepTime","CRSArrTime","DepDelay","ArrDelay","Cancelled","Diverted","Distance","CRSElapsedTime"]
frames, stats = [], []
for z in sorted(glob.glob("data/bts_*.zip")):
    zf = zipfile.ZipFile(z); name = [n for n in zf.namelist() if n.endswith(".csv")][0]
    d = pd.read_csv(zf.open(name), usecols=cols, low_memory=False)
    stats.append(dict(file=os.path.basename(z), rows=len(d), cancelled=int(d.Cancelled.sum()), diverted=int(d.Diverted.sum()), no_arrdelay=int(d.ArrDelay.isna().sum())))
    d["FlightDate"] = pd.to_datetime(d.FlightDate)
    for c in ("CRSDepTime", "CRSArrTime"): d[c] = d[c].fillna(0).astype(int).clip(upper=2359)
    d["dep_local"] = d.FlightDate + pd.to_timedelta(d.CRSDepTime // 100, unit="h") + pd.to_timedelta(d.CRSDepTime % 100, unit="m")
    frames.append(d)
bts = pd.concat(frames, ignore_index=True)
bts["dep_hour_local"] = bts.dep_local.dt.floor("h")
# congestion: scheduled departures from the same origin in the same local hour (all scheduled flights, incl. later cancelled)
bts["origin_hour_flights"] = bts.groupby(["Origin", "dep_hour_local"]).Origin.transform("size").astype("int16")
bts["dest_hour_arrivals"] = bts.groupby(["Dest", bts.FlightDate + pd.to_timedelta(bts.CRSArrTime // 100, unit="h")]).Dest.transform("size").astype("int16")
print(pd.DataFrame(stats)); print(len(bts), "scheduled flights", bts.FlightDate.min().date(), "to", bts.FlightDate.max().date(), "|", bts.Origin.nunique(), "origin airports")

## 2. Airports (OurAirports `airport_master`) and time zones

In [ ]:
ap = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv", keep_default_na=False)
ap = ap[ap.type.isin(["small_airport","medium_airport","large_airport"])][["ident","type","name","latitude_deg","longitude_deg","municipality","iso_country","iata_code"]].reset_index(drop=True)
print("airport_master rows:", len(ap), "(lookup for the app; training covers only the US airports below)")
am = ap[ap.iata_code != ""].copy(); am["r"] = am.type.map({"large_airport":0,"medium_airport":1,"small_airport":2}); am = am.sort_values("r").drop_duplicates("iata_code").set_index("iata_code")
cnt = bts.Origin.value_counts().add(bts.Dest.value_counts(), fill_value=0)
airports = [a for a in cnt[cnt >= MIN_FLIGHTS].index if a in am.index]
print(len(airports), "airports with >=", MIN_FLIGHTS, "flights")
from timezonefinder import TimezoneFinder
tf = TimezoneFinder(); tz = {a: tf.timezone_at(lat=am.loc[a,"latitude_deg"], lng=am.loc[a,"longitude_deg"]) for a in airports}

## 3. Historical hourly weather (Open-Meteo archive, ECMWF IFS model, keyless)
Same model, variables and units are used at prediction time (forecast API), so training and inference weather share one schema: temperature C, apparent temperature C, surface pressure hPa, humidity %, wind m/s, wind direction deg, gusts m/s, visibility m, precipitation mm in the hour, cloud cover %. Visibility from this model exists from Oct 2025 onward, which is why the window starts in Oct 2025.

In [ ]:
HV = "temperature_2m,apparent_temperature,surface_pressure,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,visibility,precipitation,cloud_cover"
START = (pd.Timestamp(bts.FlightDate.min()) - pd.Timedelta(days=1)).strftime("%Y-%m-%d"); END = (pd.Timestamp(bts.FlightDate.max()) + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
def fetch(a):
    fn = f"data/wx/{a}.parquet"
    if os.path.exists(fn): return
    for attempt in range(8):
        try:
            r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=dict(latitude=am.loc[a,"latitude_deg"], longitude=am.loc[a,"longitude_deg"], start_date=START, end_date=END, hourly=HV, timezone="UTC", models="ecmwf_ifs"), timeout=120)
            if r.status_code == 200:
                j = r.json()
                if "hourly" in j:
                    h = pd.DataFrame(j["hourly"]); h["time"] = pd.to_datetime(h["time"]); h["airport"] = a; h.to_parquet(fn); return
            time.sleep(20 if r.status_code == 429 else 5)
        except Exception: time.sleep(5)
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(4) as ex: list(ex.map(fetch, airports))
missing = [a for a in airports if not os.path.exists(f"data/wx/{a}.parquet")]
print("weather files:", len(airports) - len(missing), "| missing:", missing)
wx = pd.concat([pd.read_parquet(f"data/wx/{a}.parquet") for a in airports if a not in missing], ignore_index=True)

## 4. Merge flights + time-matched weather (origin at departure hour, destination at scheduled-arrival hour)

In [ ]:
d = bts[(bts.Cancelled == 0) & (bts.Diverted == 0) & bts.ArrDelay.notna() & bts.CRSElapsedTime.notna()].copy(); n_completed = len(d)
wair = set(wx.airport.unique()); d = d[d.Origin.isin(wair) & d.Dest.isin(wair)].copy()
parts = []
for o, g in d.groupby("Origin"):
    parts.append(g.dep_local.dt.tz_localize(tz[o], ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC"))
d["dep_utc"] = pd.concat(parts).reindex(d.index); d = d[d.dep_utc.notna()].copy()
d["arr_utc"] = d.dep_utc + pd.to_timedelta(d.CRSElapsedTime, unit="m")
d["dep_hr_utc"] = d.dep_utc.dt.floor("h").dt.tz_localize(None); d["arr_hr_utc"] = d.arr_utc.dt.floor("h").dt.tz_localize(None)
wx2 = wx.rename(columns={"temperature_2m":"temperature_c","apparent_temperature":"feels_like_c","surface_pressure":"pressure_hpa","relative_humidity_2m":"humidity_pct","wind_direction_10m":"wind_deg","precipitation":"rain_occurrence","cloud_cover":"cloud_cover_pct"})
wx2["wind_strength"] = wx2.wind_speed_10m / 3.6; wx2["wind_gust_ms"] = wx2.wind_gusts_10m / 3.6
W = ["temperature_c","feels_like_c","pressure_hpa","humidity_pct","wind_strength","wind_deg","visibility","rain_occurrence","cloud_cover_pct","wind_gust_ms"]
d = d.merge(wx2[["airport","time"]+W].rename(columns={"airport":"Origin","time":"dep_hr_utc"}), on=["Origin","dep_hr_utc"], how="left")
d = d.merge(wx2[["airport","time"]+W].rename(columns={"airport":"Dest","time":"arr_hr_utc", **{c: "dest_"+c for c in W}}), on=["Dest","arr_hr_utc"], how="left")
d = d[d.temperature_c.notna() & d.dest_temperature_c.notna()]
print("completed flights:", n_completed, "| with weather at both ends:", len(d))

## 5. Cleaning step (this is the cell the old notebook was missing)
Creates `df_clean`, sets `delay_binary` (arrival delay > 15 min) and saves the `flights_clean` table in SQLite, which the later cells and the app read. Cancelled/diverted flights and flights without a measured arrival delay were already excluded above (counts printed there).

In [ ]:
d = d.rename(columns={"Origin":"origin","Dest":"destination","Reporting_Airline":"airline_id","ArrDelay":"target_delay_minutes","FlightDate":"flight_date"})
d["scheduled_departure_utc"] = d.dep_utc.dt.tz_convert("UTC"); d["scheduled_arrival_utc"] = d.arr_utc
d["estimated_path_length_km"] = (d.Distance * 1.609344).round(2); d["route"] = d.origin + "_" + d.destination
d["departure_hour"] = d.scheduled_departure_utc.dt.hour; d["departure_minute"] = d.scheduled_departure_utc.dt.minute; d["departure_hour_float"] = d.departure_hour + d.departure_minute / 60.0
d["day_of_week"] = d.scheduled_departure_utc.dt.dayofweek; d["month"] = d.scheduled_departure_utc.dt.month; d["day_of_month"] = d.scheduled_departure_utc.dt.day; d["is_weekend"] = d.day_of_week.isin([5, 6]).astype(int)
d["part_of_day"] = np.select([(d.departure_hour>=5)&(d.departure_hour<12),(d.departure_hour>=12)&(d.departure_hour<17),(d.departure_hour>=17)&(d.departure_hour<21)], ["morning","afternoon","evening"], "night")
d["local_departure_hour_float"] = d.dep_local.dt.hour + d.dep_local.dt.minute / 60.0
d["high_wind_flag"] = (d.wind_strength >= 10).astype(int); d["rain_flag"] = (d.rain_occurrence > 0).astype(int); d["low_visibility_flag"] = (d.visibility < 5000).astype(int)
d["weather_severity_score"] = d.high_wind_flag + d.rain_flag + d.low_visibility_flag
d["dest_weather_severity_score"] = (d.dest_wind_strength >= 10).astype(int) + (d.dest_rain_occurrence > 0).astype(int) + (d.dest_visibility < 5000).astype(int)
# ---- minimal cleaning ----
df_clean = d[d.target_delay_minutes.notna()].drop_duplicates(subset=["flight_date","airline_id","Flight_Number_Reporting_Airline","origin","destination","scheduled_departure_utc"]).copy()
df_clean["delay_binary"] = (df_clean.target_delay_minutes > 15).astype(int)
df_clean = df_clean.sample(min(SAMPLE, len(df_clean)), random_state=42).sort_values("scheduled_departure_utc").reset_index(drop=True)
con_work = sqlite3.connect("data/aerodelay_work.db")
df_clean[["flight_date","origin","destination","route","airline_id","scheduled_departure_utc","target_delay_minutes","delay_binary","estimated_path_length_km"]].astype({"scheduled_departure_utc":str,"flight_date":str}).to_sql("flights_clean", con_work, if_exists="replace", index=False)
print("flights_clean rows:", len(df_clean), "| airports:", df_clean.origin.nunique(), "| carriers:", df_clean.airline_id.nunique(), "| routes:", df_clean.route.nunique())
print("date range:", df_clean.scheduled_departure_utc.min(), "->", df_clean.scheduled_departure_utc.max(), "| delayed >15 min:", round(df_clean.delay_binary.mean(), 4))

## 6. Features, time-based split, baselines and model comparison

In [ ]:
tr, va, te = split(df_clean)
print("train", len(tr), "| validation", len(va), "| test", len(te))
E = encodings(tr); tr, M = oof_safe(tr); va = apply_safe(va, M); te = apply_safe(te, M)
tr, va, te = [apply_enc(x, E) for x in (tr, va, te)]
FEATS = [c for c in EXT if c not in ("month", "day_of_month")]   # shipped feature set (calendar month/day dropped: only 10 months of history)
med = {c: float(tr[c].median()) for c in FEATS}
for x in (tr, va, te):
    for c in FEATS: x[c] = x[c].fillna(med[c])
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
ytr = tr.target_delay_minutes.clip(-60, 300)
rm = lambda y, p: dict(MAE=round(float(mean_absolute_error(y, p)), 3), RMSE=round(float(np.sqrt(mean_squared_error(y, p))), 3))
res = {}
for nm, df in (("validation", va), ("test", te)):
    y = df.target_delay_minutes
    res[("baseline: always median", nm)] = rm(y, np.full(len(y), tr.target_delay_minutes.median())); res[("baseline: always mean", nm)] = rm(y, np.full(len(y), tr.target_delay_minutes.mean()))
for name, mk in {"RandomForest": lambda: RandomForestRegressor(n_estimators=120, max_depth=14, min_samples_leaf=50, max_features=0.5, max_samples=0.5, n_jobs=-1, random_state=42),
                 "HistGB squared error": lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=42),
                 "HistGB absolute error": lambda: HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=42)}.items():
    m = mk().fit(tr[FEATS], ytr)
    for nm, df in (("validation", va), ("test", te)): res[(name, nm)] = rm(df.target_delay_minutes, m.predict(df[FEATS]))
print(pd.DataFrame(res).T)
cres = {}
for name, m in {"Logistic regression": Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]), "HistGB classifier": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=42)}.items():
    m.fit(tr[FEATS], tr.delay_binary)
    for nm, df in (("validation", va), ("test", te)):
        p = m.predict_proba(df[FEATS])[:, 1]; cres[(name, nm)] = dict(AUC=round(roc_auc_score(df.delay_binary, p), 4), PR_AUC=round(average_precision_score(df.delay_binary, p), 4))
cres[("baseline: always on-time", "test")] = dict(AUC=0.5, accuracy=round(float((te.delay_binary == 0).mean()), 4))
print(pd.DataFrame(cres).T)

## 7. Final fit on train+validation, one more evaluation on the untouched test months, SHAP

In [ ]:
import shap, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fit = pd.concat([tr, va]); fit_raw = pd.concat([split(df_clean)[0], split(df_clean)[1]])
E = encodings(fit_raw); fit, M = oof_safe(fit_raw); te = apply_safe(split(df_clean)[2], M); fit = apply_enc(fit, E); te = apply_enc(te, E)
med = {c: float(fit[c].median()) for c in FEATS}
for x in (fit, te):
    for c in FEATS: x[c] = x[c].fillna(med[c])
rf = RandomForestRegressor(n_estimators=120, max_depth=14, min_samples_leaf=50, max_features=0.5, max_samples=0.5, n_jobs=-1, random_state=42).fit(fit[FEATS], fit.target_delay_minutes.clip(-60, 300))
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=200, l2_regularization=1.0, random_state=42).fit(fit[FEATS], fit.delay_binary)
p = rf.predict(te[FEATS]); pr = clf.predict_proba(te[FEATS])[:, 1]; yb = te.delay_binary
THR = 0.20   # risk threshold that maximised F1 on the validation months in our run; re-tune on validation if you change data
final = dict(rf_test=rm(te.target_delay_minutes, p), median_baseline_test=rm(te.target_delay_minutes, np.full(len(te), fit.target_delay_minutes.median())),
             AUC=float(roc_auc_score(yb, pr)), PR_AUC=float(average_precision_score(yb, pr)), accuracy_at_0_5=float(accuracy_score(yb, pr >= .5)), F1_at_0_5=float(f1_score(yb, pr >= .5)),
             accuracy_at_thr=float(accuracy_score(yb, pr >= THR)), F1_at_thr=float(f1_score(yb, pr >= THR)), majority_accuracy=float((yb == 0).mean()), test_delay_rate=float(yb.mean()), train_delay_rate=float(fit.delay_binary.mean()))
print(json.dumps(final, indent=1))
sample = te.sample(min(2000, len(te)), random_state=1)[FEATS]
sv = shap.TreeExplainer(rf).shap_values(sample)
imp = pd.DataFrame({"feature": FEATS, "mean_abs_shap_minutes": np.abs(sv).mean(0)}).sort_values("mean_abs_shap_minutes", ascending=False).reset_index(drop=True)
print(imp.head(15).round(3))
top = imp.head(15).iloc[::-1]; fig, ax = plt.subplots(figsize=(11, 8)); ax.barh(top.feature, top.mean_abs_shap_minutes); ax.set_xlabel("Mean |SHAP value| (minutes)"); ax.set_title("AeroDelay AI (retrained): top 15 drivers of predicted arrival delay"); plt.tight_layout(); plt.savefig("out/shap_top_drivers.png", dpi=150); plt.show()

## 8. Save artifacts (same file names as the repo) and the SQLite DB

In [ ]:
g = d.copy()
cong_o = {f"{a}|{h}|{w}": float(v) for (a, h, w), v in g.assign(h=g.local_departure_hour_float.astype(int)).groupby(["origin","h","is_weekend"]).origin_hour_flights.median().round(1).items()}
g["dur"] = (pd.to_datetime(g.scheduled_arrival_utc, utc=True) - pd.to_datetime(g.scheduled_departure_utc, utc=True)).dt.total_seconds() / 60
route_dur = g.groupby("route").dur.median().round(0).to_dict(); bfit = [float(x) for x in np.polyfit(g.estimated_path_length_km, g.dur, 1)]
parts = [pd.to_datetime(x.scheduled_arrival_utc, utc=True).dt.tz_convert(tz[k]).dt.hour for k, x in g.groupby("destination")]
g["arr_h"] = pd.concat(parts).reindex(g.index); g["arr_wk"] = pd.to_datetime(g.scheduled_arrival_utc, utc=True).dt.dayofweek.isin([5, 6]).astype(int)
cong_d = {f"{a}|{h}|{w}": float(v) for (a, h, w), v in g.groupby(["destination","arr_h","arr_wk"]).dest_hour_arrivals.median().round(1).items()}
art = dict(model_feature_columns=FEATS, train_medians=med, **M, split_method="time_based", classifier_type="hist_gradient_boosting", risk_threshold_f1=THR, airport_tz=tz, congestion_origin=cong_o, congestion_dest=cong_d,
           route_block_minutes=route_dur, block_minutes_fit=bfit, known_carriers=sorted(E["carrier_map"]), metrics=dict(rf_test=final["rf_test"], base_median_test=final["median_baseline_test"], clf_test=dict(AUC=final["AUC"])))
joblib.dump(rf, "out/aerodelay_rf_model.pkl", compress=3); joblib.dump(clf, "out/aerodelay_log_model.pkl", compress=3); joblib.dump(art, "out/aerodelay_artifacts.pkl", compress=3); joblib.dump(E, "out/aerodelay_encoding_maps.pkl", compress=3)
if os.path.exists("out/aerodelay.db"): os.remove("out/aerodelay.db")
con = sqlite3.connect("out/aerodelay.db")
fc = df_clean.sample(min(100000, len(df_clean)), random_state=7)[["flight_date","origin","destination","route","airline_id","scheduled_departure_utc","target_delay_minutes","delay_binary","estimated_path_length_km"]].astype({"scheduled_departure_utc": str, "flight_date": str})
fc.to_sql("flights_clean", con, index=False); ap.to_sql("airport_master", con, index=False)
mm = pd.DataFrame([("train_rows", len(fit)), ("test_rows", len(te)), ("rf_mae", final["rf_test"]["MAE"]), ("rf_rmse", final["rf_test"]["RMSE"]), ("baseline_median_mae", final["median_baseline_test"]["MAE"]), ("clf_auc", final["AUC"]), ("clf_accuracy_at_thr", final["accuracy_at_thr"]), ("clf_f1_at_thr", final["F1_at_thr"])], columns=["metric", "value"])
mm.to_sql("model_metrics", con, index=False); imp.to_sql("shap_importance", con, index=False); con.commit(); con.close()
for f in sorted(os.listdir("out")): print(f, round(os.path.getsize("out/" + f) / 1e6, 1), "MB")

## 9. Prediction path used by the app: forecast weather
`aerodelay_infer.py` (written below) builds the same features at prediction time. Weather for the chosen airport/date/time comes from the **Open-Meteo forecast API** (same ECMWF IFS model and units as training) for the next ~15 days, the archive for past dates, and a clearly labelled **seasonal analog** (same date last year, +/-3 days) beyond the forecast window. OpenWeather current weather is used only if Open-Meteo is unreachable and `OPENWEATHER_API_KEY` is set in the environment. Times are entered as LOCAL time at the departure airport.

In [ ]:
from pathlib import Path
Path('out/aerodelay_infer.py').write_text(r'''"""AeroDelay AI inference helpers (shared by the Gradio app and the notebook).

Weather path for a prediction (all sources return the SAME schema/units as the training weather):
  1. Open-Meteo forecast API (ECMWF IFS, keyless) for -7 .. +15 days from today.   -> label "forecast"
  2. Open-Meteo archive API (same ECMWF IFS model) for older dates.                   -> label "historical"
  3. Beyond the forecast window: same calendar date one year earlier (+/-3 days)
     from the archive API, averaged by hour.                                          -> label "seasonal analog (NOT a forecast)"
  4. If Open-Meteo is unreachable: OpenWeather current weather, only if OPENWEATHER_API_KEY is set
     in the environment (key is never printed/logged).                                 -> label "OpenWeather current"
  5. Last resort: training medians.                                                    -> label "weather forecast unavailable, typical values used"
"""
import os, math, json, sqlite3, time
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import numpy as np, pandas as pd, requests, joblib

HERE = os.path.dirname(os.path.abspath(__file__))
OM_VARS = "temperature_2m,apparent_temperature,surface_pressure,relative_humidity_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,visibility,precipitation,cloud_cover"
FORECAST_DAYS = 15

def load(path=HERE):
    A = joblib.load(os.path.join(path, "aerodelay_artifacts.pkl"))
    rf = joblib.load(os.path.join(path, "aerodelay_rf_model.pkl"))
    clf = joblib.load(os.path.join(path, "aerodelay_log_model.pkl"))
    con = sqlite3.connect(os.path.join(path, "aerodelay.db"))
    airports = pd.read_sql("SELECT * FROM airport_master", con)
    fc = pd.read_sql("SELECT origin,destination,route,airline_id FROM flights_clean", con)
    con.close()
    for c in ("iata_code", "ident", "name"):
        airports[c + "_u"] = airports[c].astype(str).str.upper().str.strip()
    return dict(A=A, rf=rf, clf=clf, airports=airports, flights=fc)

# ---------------------------------------------------------------- airports / time
def resolve_airport(text, ctx):
    t = str(text or "").strip().upper()
    if not t: raise ValueError("Please enter an airport (IATA like ATL, ICAO like KATL, or part of the name).")
    ap = ctx["airports"]
    for col in ("iata_code_u", "ident_u", "name_u"):
        m = ap[ap[col] == t]
        if not m.empty: return m.sort_values("type", key=lambda s: s.map({"large_airport": 0, "medium_airport": 1, "small_airport": 2}).fillna(3)).iloc[0].to_dict()
    m = ap[ap["name_u"].str.contains(t, na=False, regex=False)]
    if not m.empty:
        m = m.assign(_r=m["type"].map({"large_airport": 0, "medium_airport": 1, "small_airport": 2}).fillna(3), _i=(m["iata_code_u"] == "").astype(int))
        return m.sort_values(["_i", "_r"]).iloc[0].to_dict()
    raise ValueError(f"Could not find airport '{text}'.")

def airport_code(row):
    return row["iata_code_u"] if row.get("iata_code_u") else row["ident_u"]

def airport_tz(row, A):
    code = airport_code(row)
    if code in A["airport_tz"]: return A["airport_tz"][code], "training table"
    try:
        from timezonefinder import TimezoneFinder
        z = TimezoneFinder().timezone_at(lat=float(row["latitude_deg"]), lng=float(row["longitude_deg"]))
        if z: return z, "timezonefinder"
    except Exception: pass
    off = round(float(row["longitude_deg"]) / 15.0)
    return f"Etc/GMT{-off:+d}".replace("+-", "-") if off else "UTC", "approximate (longitude/15)"

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(a))

def parse_time(s):
    s = str(s or "").strip().upper().replace(" ", "")
    if not s: raise ValueError("Please enter a departure time like 14:30 or 2:30PM.")
    for fmt in ("%H:%M", "%H%M", "%I:%M%p", "%I%p"):
        try:
            t = datetime.strptime(s, fmt); return t.hour, t.minute
        except ValueError: pass
    raise ValueError("Invalid time. Use 14:30, 1430 or 2:30PM (local time at the departure airport).")

# ---------------------------------------------------------------- weather
WX_UNAVAILABLE = "weather forecast unavailable, typical values used"
WX_COLS = ["temperature_c", "feels_like_c", "pressure_hpa", "humidity_pct", "wind_strength", "wind_deg", "visibility", "rain_occurrence", "cloud_cover_pct", "wind_gust_ms"]

def _om_to_schema(h):
    """Open-Meteo hourly frame -> training schema/units (km/h -> m/s, precipitation mm/h)."""
    return pd.DataFrame({"time": pd.to_datetime(h["time"]), "temperature_c": h["temperature_2m"], "feels_like_c": h["apparent_temperature"], "pressure_hpa": h["surface_pressure"],
        "humidity_pct": h["relative_humidity_2m"], "wind_strength": pd.Series(h["wind_speed_10m"], dtype=float) / 3.6, "wind_deg": h["wind_direction_10m"], "visibility": h["visibility"],
        "rain_occurrence": h["precipitation"], "cloud_cover_pct": h["cloud_cover"], "wind_gust_ms": pd.Series(h["wind_gusts_10m"], dtype=float) / 3.6})

def _om(url, params, retries=2):
    """Open-Meteo request with 2 retries (short backoff) on 5xx / network errors. Errors never include the URL."""
    last = "unknown error"
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, params=params, timeout=25)
            if r.status_code >= 500:
                last = f"HTTP {r.status_code}"
            elif r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}")
            else:
                j = r.json()
                if "hourly" not in j: raise RuntimeError(str(j.get("reason", "no hourly data"))[:60])
                return _om_to_schema(j["hourly"])
        except (requests.ConnectionError, requests.Timeout) as e:
            last = type(e).__name__
        if attempt < retries: time.sleep(1.0 * (attempt + 1))
    raise RuntimeError(last)

def get_weather(lat, lon, when_utc, A):
    """when_utc: tz-aware UTC datetime. Returns (dict in training schema, label)."""
    hour = pd.Timestamp(when_utc).tz_convert("UTC").tz_localize(None).floor("h")
    now = pd.Timestamp.now(tz="UTC").tz_localize(None).floor("h")
    base = dict(latitude=lat, longitude=lon, hourly=OM_VARS, timezone="UTC", models="ecmwf_ifs")
    days_ahead = (hour - now).total_seconds() / 86400
    try:
        if -7 <= days_ahead <= FORECAST_DAYS - 0.5:
            df = _om("https://api.open-meteo.com/v1/forecast", {**base, "past_days": 7, "forecast_days": FORECAST_DAYS + 1})
            row = df[df.time == hour]
            if not row.empty and row[WX_COLS[:6]].notna().all(axis=1).iloc[0]:
                return _fill(row.iloc[0], A), ("current conditions (Open-Meteo, now)" if abs(days_ahead) < 1/24 else ("forecast (Open-Meteo ECMWF IFS)" if days_ahead > 0 else "recent observed/analysis (Open-Meteo)"))
        elif days_ahead < -7:
            d0 = hour.strftime("%Y-%m-%d")
            df = _om("https://archive-api.open-meteo.com/v1/archive", {**base, "start_date": d0, "end_date": d0})
            row = df[df.time == hour]
            if not row.empty and row[WX_COLS[:6]].notna().all(axis=1).iloc[0]:
                return _fill(row.iloc[0], A), "historical (Open-Meteo archive)"
        # beyond forecast window (or empty hour): seasonal analog
        an = hour - pd.Timedelta(days=365)
        df = _om("https://archive-api.open-meteo.com/v1/archive", {**base, "start_date": (an - pd.Timedelta(days=3)).strftime("%Y-%m-%d"), "end_date": (an + pd.Timedelta(days=3)).strftime("%Y-%m-%d")})
        df = df[df.time.dt.hour == hour.hour]
        if df.empty: raise RuntimeError("no analog data")
        s = df[WX_COLS].astype(float).mean()
        s["wind_deg"] = df["wind_deg"].astype(float).median()
        return _fill(s, A), "SEASONAL ANALOG: same date last year +/-3 days averaged. NOT a forecast"
    except Exception as e:
        pass
    key = os.environ.get("OPENWEATHER_API_KEY")
    if key and abs((hour - now).total_seconds()) < 86400 * 1.5:
        try:
            r = requests.get("https://api.openweathermap.org/data/2.5/weather", params={"lat": lat, "lon": lon, "appid": key, "units": "metric"}, timeout=15); r.raise_for_status(); w = r.json()
            s = pd.Series(dict(temperature_c=w["main"]["temp"], feels_like_c=w["main"]["feels_like"], pressure_hpa=w["main"].get("grnd_level", w["main"]["pressure"]), humidity_pct=w["main"]["humidity"],
                wind_strength=w.get("wind", {}).get("speed", np.nan), wind_deg=w.get("wind", {}).get("deg", np.nan), visibility=w.get("visibility", np.nan),
                rain_occurrence=w.get("rain", {}).get("1h", 0.0), cloud_cover_pct=w.get("clouds", {}).get("all", np.nan), wind_gust_ms=w.get("wind", {}).get("gust", np.nan)))
            return _fill(s, A), "current conditions (OpenWeather, Open-Meteo unavailable)"
        except Exception: pass
    return {c: A["train_medians"].get(c, 0.0) for c in WX_COLS}, WX_UNAVAILABLE

def _fill(s, A):
    out = {}
    for c in WX_COLS:
        v = s.get(c, np.nan)
        out[c] = float(v) if pd.notna(v) else float(A["train_medians"].get(c, 0.0))
    return out

def weather_flags(w, prefix=""):
    hw = int(w["wind_strength"] >= 10); rn = int(w["rain_occurrence"] > 0); lv = int(w["visibility"] < 5000)
    return hw, rn, lv

# ---------------------------------------------------------------- features
def build_row(origin, destination, date_str, time_str, carrier, ctx):
    A = ctx["A"]; warnings = []
    o = resolve_airport(origin, ctx); d = resolve_airport(destination, ctx)
    oc, dc = airport_code(o), airport_code(d)
    if oc == dc: raise ValueError("Origin and destination must be different.")
    otz, otz_src = airport_tz(o, A); dtz, _ = airport_tz(d, A)
    if otz_src.startswith("approx"): warnings.append(f"Time zone for {oc} is approximate.")
    hh, mm = parse_time(time_str)
    day = pd.to_datetime(date_str).date() if str(date_str or "").strip() else datetime.now(ZoneInfo(otz)).date()
    dep_local = datetime(day.year, day.month, day.day, hh, mm, tzinfo=ZoneInfo(otz)); dep_utc = dep_local.astimezone(timezone.utc)
    dist = haversine(float(o["latitude_deg"]), float(o["longitude_deg"]), float(d["latitude_deg"]), float(d["longitude_deg"]))
    route = f"{oc}_{dc}"
    in_o, in_d = oc in A["origin_mean_map"], dc in A["destination_mean_map"]
    if not in_o: warnings.append(f"{oc} is not in the training data (US domestic airports with 300+ flights). Origin effects use the overall average.")
    if not in_d: warnings.append(f"{dc} is not in the training data (US domestic airports with 300+ flights). Destination effects use the overall average.")
    if route not in A["route_mean_map"]: warnings.append(f"Route {route} was not seen in training. Route effect uses the overall average.")
    if dist > 6000 or not (in_o and in_d): warnings.append("Outside the US-domestic training coverage: treat this prediction as low confidence.")
    # carrier
    carrier = str(carrier or "").strip().upper(); csrc = "you entered"
    if not carrier:
        f = ctx["flights"]; sub = f[f.route == route].airline_id
        csrc = "most common on this route in training data"
        if sub.empty: sub = f[f.origin == oc].airline_id; csrc = "route not in training data; most common carrier at the origin (a guess, enter the airline if you know it)"
        if not len(sub): csrc = "unknown"
        carrier = sub.mode().iloc[0] if len(sub) else "UNKNOWN"
    if carrier not in A["carrier_mean_map"]:
        warnings.append(f"Carrier '{carrier}' is not in the training data (BTS US carriers: {', '.join(A['known_carriers'])}). Carrier effects use the overall average.")
    # weather: origin at departure hour, destination at scheduled arrival hour
    blk = A["route_block_minutes"].get(route, A["block_minutes_fit"][0] * dist + A["block_minutes_fit"][1]); arr_utc = dep_utc + timedelta(minutes=float(blk))
    wo, wo_src = get_weather(float(o["latitude_deg"]), float(o["longitude_deg"]), dep_utc, A)
    wd, wd_src = get_weather(float(d["latitude_deg"]), float(d["longitude_deg"]), arr_utc, A)
    hw, rn, lv = weather_flags(wo); dhw, drn, dlv = weather_flags({"wind_strength": wd["wind_strength"], "rain_occurrence": wd["rain_occurrence"], "visibility": wd["visibility"]})
    wk = int(dep_utc.weekday() >= 5); ldh = hh + mm / 60.0
    arr_local = arr_utc.astimezone(ZoneInfo(dtz))
    g = float(A["global_train_delay_mean"])
    f = {"departure_hour": dep_utc.hour, "departure_minute": dep_utc.minute, "departure_hour_float": dep_utc.hour + dep_utc.minute / 60.0, "day_of_week": dep_utc.weekday(), "is_weekend": wk,
         "encoded_origin": ctx["E"]["origin_map"].get(oc, -1), "encoded_destination": ctx["E"]["destination_map"].get(dc, -1), "encoded_route": ctx["E"]["route_map"].get(route, -1),
         "encoded_carrier": ctx["E"]["carrier_map"].get(carrier, -1), "encoded_part_of_day": ctx["E"]["part_of_day_map"].get(_pod(dep_utc.hour), -1), "estimated_path_length_km": dist,
         "origin_avg_delay_safe": A["origin_mean_map"].get(oc, g), "destination_avg_delay_safe": A["destination_mean_map"].get(dc, g), "route_avg_delay_safe": A["route_mean_map"].get(route, g), "carrier_avg_delay_safe": A["carrier_mean_map"].get(carrier, g),
         "origin_flight_count_safe": A["origin_count_map"].get(oc, 0), "destination_flight_count_safe": A["destination_count_map"].get(dc, 0), "route_flight_count_safe": A["route_count_map"].get(route, 0), "carrier_flight_count_safe": A["carrier_count_map"].get(carrier, 0),
         "high_wind_flag": hw, "rain_flag": rn, "low_visibility_flag": lv, "weather_severity_score": hw + rn + lv,
         "local_departure_hour_float": ldh, "origin_hour_flights": A["congestion_origin"].get(f"{oc}|{hh}|{wk}", A["train_medians"]["origin_hour_flights"]),
         "dest_hour_arrivals": A["congestion_dest"].get(f"{dc}|{arr_local.hour}|{int(arr_utc.weekday() >= 5)}", A["train_medians"]["dest_hour_arrivals"]),
         "dest_temperature_c": wd["temperature_c"], "dest_wind_strength": wd["wind_strength"], "dest_visibility": wd["visibility"], "dest_rain_occurrence": wd["rain_occurrence"], "dest_cloud_cover_pct": wd["cloud_cover_pct"], "dest_weather_severity_score": dhw + drn + dlv}
    f.update({k: wo[k] for k in WX_COLS})
    row = {c: f.get(c, A["train_medians"].get(c, 0)) for c in A["model_feature_columns"]}
    X = pd.DataFrame([row])
    meta = dict(origin=oc, destination=dc, origin_name=o["name"], destination_name=d["name"], route=route, carrier=carrier, carrier_source=csrc, dep_local=dep_local, dep_utc=dep_utc, distance_km=dist, block_minutes=float(blk),
                weather_origin=wo, weather_origin_src=wo_src, weather_dest=wd, weather_dest_src=wd_src, warnings=warnings, origin_tz=otz)
    return X, meta

def _pod(h): return "morning" if 5 <= h < 12 else "afternoon" if 12 <= h < 17 else "evening" if 17 <= h < 21 else "night"

def band(p):
    return ("LOW", "<20%") if p < 0.20 else (("ELEVATED", "20-35%") if p < 0.35 else ("HIGH", ">=35%"))

def predict(origin, destination, date_str, time_str, carrier, ctx, explainer=None, top_k=5):
    X, meta = build_row(origin, destination, date_str, time_str, carrier, ctx)
    minutes = float(ctx["rf"].predict(X)[0]); prob = float(ctx["clf"].predict_proba(X)[0][1])   # real classifier: no constant fallback
    out = dict(minutes=minutes, risk=prob, band=band(prob), meta=meta, X=X)
    if explainer is not None:
        sv = explainer.shap_values(X)[0]
        s = pd.DataFrame({"feature": X.columns, "shap_minutes": sv, "value": X.iloc[0].values}); s["abs"] = s.shap_minutes.abs()
        out["shap"] = s.sort_values("abs", ascending=False).head(top_k).drop(columns="abs").reset_index(drop=True)
        out["shap_base"] = float(np.ravel(explainer.expected_value)[0])
    return out

def setup(path=HERE):
    import shap
    ctx = load(path); ctx["E"] = joblib.load(os.path.join(path, "aerodelay_encoding_maps.pkl")); ctx["explainer"] = shap.TreeExplainer(ctx["rf"]); return ctx
''')

In [ ]:
Path('out/app.py').write_text(r'''"""AeroDelay AI (retrained) - Gradio app. Run: python app.py   (needs the files next to it)."""
import os, gradio as gr
from datetime import datetime
import aerodelay_infer as ai

CTX = ai.setup(os.path.dirname(os.path.abspath(__file__)))
M = CTX["A"]["metrics"]

NL = chr(10)          # real line breaks (no backslash escapes in the output text)
ICON = {"LOW": chr(0x1F7E2), "ELEVATED": chr(0x1F7E1), "HIGH": chr(0x1F534)}

def wx_line(w, src):
    if src == ai.WX_UNAVAILABLE:
        return ai.WX_UNAVAILABLE
    return f"{w['temperature_c']:.1f} C, wind {w['wind_strength']:.1f} m/s, visibility {w['visibility']/1000:.1f} km, rain {w['rain_occurrence']:.1f} mm/h, cloud {w['cloud_cover_pct']:.0f}% ({src})"

def run(origin, destination, date_str, time_str, carrier):
    try:
        r = ai.predict(origin, destination, date_str, time_str, carrier, CTX, CTX["explainer"])
    except Exception as e:
        return "**Could not predict:** " + str(e)[:200]
    m = r["meta"]; band, rng = r["band"]
    P = []   # paragraphs, joined with a blank line so every field renders on its own line
    P.append(f"## {ICON[band]} Delay risk: {band} ({r['risk']*100:.1f}%)")
    P.append(f"**Delay risk:** {r['risk']*100:.1f}% chance of arriving more than 15 min late. Band: {band} ({rng}).")
    P.append(f"**Typical-case delay estimate:** {max(r['minutes'],0):.1f} min (model average for flights like this; individual flights vary a lot)")
    P.append(f"**Route:** {m['origin']} - {m['origin_name']} to {m['destination']} - {m['destination_name']} ({m['distance_km']:.0f} km, about {m['block_minutes']:.0f} min scheduled)")
    P.append(f"**Departure:** {m['dep_local'].strftime('%Y-%m-%d %H:%M')} local ({m['origin_tz']}), {m['dep_utc'].strftime('%Y-%m-%d %H:%M')} UTC")
    P.append(f"**Carrier:** {m['carrier']} ({m['carrier_source']})")
    P.append("**Origin weather:** " + wx_line(m["weather_origin"], m["weather_origin_src"]))
    P.append("**Destination weather:** " + wx_line(m["weather_dest"], m["weather_dest_src"]))
    if "SEASONAL" in m["weather_origin_src"] or "SEASONAL" in m["weather_dest_src"]:
        P.append("> **Weather is a seasonal analog, not a forecast** (the date is beyond the 15-day forecast window).")
    P.append(f"**Top factors** (SHAP: minutes added or removed from the model average of {r['shap_base']:.1f} min):" + NL + NL + NL.join(f"- **{row.feature}** = {row.value:.3g}: {row.shap_minutes:+.2f} min" for row in r["shap"].itertuples()))
    if m["warnings"]:
        P.append("**Warnings:**" + NL + NL + NL.join("- " + w for w in m["warnings"]))
    P.append(f"*Held-out test (Jun-Jul 2026, US domestic): risk AUC {M['clf_test']['AUC']:.2f}; minutes MAE {M['rf_test']['MAE']:.1f} vs {M['base_median_test']['MAE']:.1f} for always guessing the median. Risk bands: LOW below 20%, ELEVATED 20-35%, HIGH 35% and up.*")
    return (NL + NL).join(P)

demo = gr.Interface(fn=run,
    inputs=[gr.Textbox(value="ATL", label="Origin airport (IATA, ICAO or name)"), gr.Textbox(value="LAX", label="Destination airport"),
            gr.Textbox(value=datetime.now().strftime("%Y-%m-%d"), label="Departure date (local, YYYY-MM-DD; any date)"), gr.Textbox(value="14:25", label="Scheduled departure time (LOCAL time at origin, e.g. 14:25)"),
            gr.Textbox(value="DL", label="Airline code (optional, e.g. DL, AA, UA, WN)")],
    outputs=gr.Markdown(label="Prediction"), title="AeroDelay AI",
    description="Predicts delay risk for US domestic flights. Trained on 10 months of BTS on-time data with time-matched weather. Uses forecast weather for the chosen airport and time.",
    examples=[["ATL", "LAX", datetime.now().strftime("%Y-%m-%d"), "14:25", "DL"], ["JFK", "MAD", datetime.now().strftime("%Y-%m-%d"), "14:30", ""]], flagging_mode="never")
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", 7860)), share=os.environ.get("GRADIO_SHARE") == "1")
''')

In [ ]:
import sys; sys.path.insert(0, "out")
import importlib, aerodelay_infer as ai; importlib.reload(ai)
from datetime import datetime, timedelta
ctx = ai.setup("out"); tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
for case in [("ATL","LAX",tomorrow,"14:25","DL"), ("ATL","LAX",tomorrow,"21:50","DL"), ("JFK","MAD",tomorrow,"14:30","")]:
    r = ai.predict(*case, ctx, ctx["explainer"]); m = r["meta"]
    print(case, "-> risk %.1f%% (%s) | typical delay %.1f min | weather: %s" % (r["risk"] * 100, r["band"][0], max(r["minutes"], 0), m["weather_origin_src"]))
    for w in m["warnings"]: print("   WARNING:", w)

## 10. Run the app
In a terminal: `cd aerodelay_retrain/out && python app.py` (set `GRADIO_SHARE=1` for a public link). In Colab, run the cell below.

In [ ]:
# import runpy; os.environ["GRADIO_SHARE"] = "1"; os.chdir("out"); runpy.run_path("app.py", run_name="__main__")